<a href="https://colab.research.google.com/github/skyadav1989/prompt-engineering/blob/main/Understanding_LLM_Internals_with_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🚀 What is prompt engineering
Prompt engineering is the art and science of crafting inputs (prompts) to large language models (LLMs) to guide them toward generating desired and higher-quality outputs.

Its usage involves carefully designing prompts to elicit specific behaviors or information from the model, such as generating creative content, answering questions, summarizing text, or performing translations.

The benefits include improved accuracy, relevance, and creativity of LLM outputs, reduced 'hallucinations' or irrelevant responses, and more efficient use of the model for various tasks, ultimately leading to better task performance and user experience.



```
# This is formatted as code
```

# 🚀 Understanding LLM Internals with Gemini

This notebook demonstrates key Large Language Model (LLM) concepts using Google's Gemini API.

Topics covered:
- Next token prediction behavior
- Temperature & randomness
- Hallucination demonstration
- Structured output prompting
- Context window effects
- Deterministic vs creative generation

Run each section step by step to observe model behavior.

## 1️⃣ Install Dependencies

We install the official Google Generative AI SDK to access Gemini models.

In [ ]:
!pip install -q google-generativeai

## 2️⃣ Setup Gemini API Key

Get your API key from: https://aistudio.google.com/app/apikey

Replace `YOUR_GEMINI_API_KEY` below.

In [ ]:
import google.generativeai as genai
from google.colab import userdata



genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

for model in genai.list_models():
    print("Name:", model.name)
    print("Supported Methods:", model.supported_generation_methods)
    print("-" * 50)

Name: models/gemini-2.5-flash
Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
Name: models/gemini-2.5-pro
Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
Name: models/gemini-2.0-flash
Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
Name: models/gemini-2.0-flash-001
Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
Name: models/gemini-2.0-flash-exp-image-generation
Supported Methods: ['generateContent', 'countTokens', 'bidiGenerateContent']
--------------------------------------------------
Name: models/gemini-2.0-flash-lite-001
Supported Methods: ['generateContent', 'countTokens'

## 3️⃣ Load Gemini Model

We load the Gemini 1.5 Pro model for experimentation.

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash-lite")



## 4️⃣ Next Token Prediction Demonstration and metadata


*Metadata Field	Meaning*
* prompt_token_count - 	Tokens in input
* candidates_token_count	- Tokens in output
* total_token_count	- Total billed tokens

In [ ]:
response = model.generate_content("The sky is")

print(response.text)
print(response.usage_metadata)

response = model.generate_content("The sky is",generation_config={
        "max_output_tokens": 50
    })

print("Output after max token description\n\n")
print(response.text)

The sky is...

This is a great, open-ended prompt! Here are a few ways to complete that sentence, depending on the mood and context you're going for:

**Descriptive & Observational:**

*   The sky is **blue**.
*   The sky is **cloudy**.
*   The sky is **clear**.
*   The sky is **full of stars**.
*   The sky is **a vibrant orange and pink** as the sun sets.
*   The sky is **a deep, inky black** with a sliver of moon.
*   The sky is **streaked with white contrails**.

**Metaphorical & Evocative:**

*   The sky is **limitless**.
*   The sky is **a canvas for dreams**.
*   The sky is **a vast, unknowable ocean**.
*   The sky is **where hopes take flight**.
*   The sky is **a silent observer of our lives**.
*   The sky is **a reminder of the grand scale of things**.

**Emotional & Personal:**

*   The sky is **calming**.
*   The sky is **inspiring**.
*   The sky is **a little bit sad today**.
*   The sky is **full of promise**.
*   The sky is **making me feel small**.

**Action-Oriented:**


# 📘 Gemini `generation_config` Complete Guide

In Gemini, `generation_config` controls how the model generates output.  
It allows you to manage creativity, stability, token length, and sampling behavior.

---

## 🔧 Parameters Explained

### 1️⃣ temperature
Controls randomness in output.

- `0` → Fully deterministic (same output every time)
- `0.2 – 0.4` → Stable (good for APIs)
- `0.7+` → Creative and varied

---

### 2️⃣ top_p (Nucleus Sampling)

The model selects from the **smallest group of tokens whose cumulative probability reaches `top_p`**.

- Lower value → Safer, more focused output  
- Higher value → More diverse output  

Example:
If token probabilities are:

| Token | Probability |
|--------|------------|
| cooling | 0.40 |
| comfort | 0.25 |
| climate | 0.15 |
| breeze | 0.10 |
| potato | 0.01 |

If `top_p = 0.7`, the model selects:

cooling (0.40)  
+ comfort (0.25)  
+ climate (0.15) = 0.80 cumulative  

It ignores lower probability tokens.

---

### 3️⃣ top_k

Limits sampling to the **top K most probable tokens**.

- Fixed number
- More strict than `top_p`

Difference:

| Parameter | Type |
|------------|--------|
| top_k | Fixed token count |
| top_p | Dynamic probability threshold |

---

### 4️⃣ max_output_tokens

Hard limit on response length.

- Prevents long outputs
- Controls API cost
- Important for structured JSON

---

### 5️⃣ candidate_count

Number of output variations generated.

Useful for:
- Self-consistency
- Choosing best response

---

### 6️⃣ stop_sequences

Stops generation when specific string appears.

Example:


Useful for:
- Structured outputs
- Chat simulations

---

### 7️⃣ presence_penalty (Model Dependent)

Encourages introduction of new topics.

---

### 8️⃣ frequency_penalty (Model Dependent)

Reduces repetition of words or phrases.

---

# 🚀 Production Recommendations

For Stable API Systems (Magento / FastAPI):

```python
{
  "temperature": 0.2,
  "top_p": 0.8,
  "max_output_tokens": 150
}

In [ ]:

generation_config = {
    "temperature": 0.4,
    "top_p": 0.85,
    "top_k": 40,
    "max_output_tokens": 150,
    "candidate_count": 1,
    "stop_sequences": ["END"]
}

response = model.generate_content(
    "Write a essay on horse riding adventure",
    generation_config=generation_config
)

print(response.text)
print(response.usage_metadata)

#Example 2 with candiate count 3

generation_config = {
    "temperature": 0.4,
    "top_p": 0.85,
    "top_k": 40,
    "max_output_tokens": 150,
    "candidate_count": 3,
    "stop_sequences": ["END"]
}

response = model.generate_content(
    "Generate an SEO product description for 1.5 Ton Inverter AC",
    generation_config=generation_config
)

print(response)



## The Wind in My Mane: An Essay on the Thrill of Horse Riding Adventure

The rhythmic thud of hooves against the earth, the scent of sun-baked leather and horse sweat, the exhilarating feeling of a powerful creature responding to your every subtle cue – these are the sensory hallmarks of a horse riding adventure. It’s more than just a sport; it’s a partnership, a journey into the wild, and a profound connection with a magnificent animal that unlocks a unique brand of freedom.

The essence of horse riding adventure lies in its inherent unpredictability and the embrace of the unknown. Unlike the controlled environments of many recreational pursuits, the open trail, the winding forest path, or the vast expanse of a meadow presents a dynamic
prompt_token_count: 8
candidates_token_count: 150
total_token_count: 158

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
         

## 5️⃣ Temperature Experiment

Temperature controls randomness.
- Low temperature = deterministic
- High temperature = creative/random

In [ ]:
low = model.generate_content(
    "Write 30 words essay on dog",
    generation_config={"temperature": 0.1}
)

high = model.generate_content(
    "Write 30 words essay on dog",
    generation_config={"temperature": 0.9}
)

print("Low Temperature:\n", low.text)
print("\nHigh Temperature:\n", high.text)

Low Temperature:
 Dogs are our most loyal companions, offering unconditional love and endless joy. Their wagging tails and warm snuggles brighten every day. Truly, they are man's best friend, enriching our lives profoundly.

High Temperature:
 Dogs are loyal, loving companions, bringing immense joy. Their unconditional affection and playful spirit brighten our lives daily. Truly, they are humanity's best friend, enriching every moment with their devoted presence.


# Top p explanation
it predicts the next token using probabilities.


In [ ]:

response = model.generate_content(
    "Complete the sentence in a maximum 10 words. Sentence : The ac is best for ",
    generation_config={
        "temperature": 0.7,
        "top_p": 0.8
    }
)

print(response.text)

NotFound: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-2.5-lite is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

## 6️⃣ Hallucination Demonstration

LLMs predict patterns, not facts.
If asked about unknown future events, they may fabricate answers.

In [ ]:
response = model.generate_content("Who won IPL in 2027?")
print(response.text)

The IPL in 2027 has not happened yet!

Since it's currently 2024, the 2027 season is still a few years in the future. We'll have to wait until then to find out who the winner will be.


## 7️⃣ Structured Output Prompting

We force the model to return structured JSON output.
Lower temperature improves consistency.

In [ ]:
prompt = """
Return output in this exact JSON format:
{
  "title": string,
  "seo_keywords": list,
  "meta_description": string
}

Product: 1.5 Ton 5 Star Inverter Split AC
"""

response = model.generate_content(
    prompt,
    generation_config={"temperature": 0.2}
)

print(response.text)

```json
{
  "title": "1.5 Ton 5 Star Inverter Split AC",
  "seo_keywords": [
    "1.5 Ton AC",
    "5 Star Inverter AC",
    "Split AC",
    "Energy Efficient AC",
    "Inverter Split AC",
    "1.5 Ton Split AC",
    "Best Inverter AC",
    "Power Saving AC",
    "Air Conditioner 1.5 Ton",
    "AC for Home"
  ],
  "meta_description": "Experience superior cooling and significant energy savings with our 1.5 Ton 5 Star Inverter Split AC. Enjoy powerful, quiet, and efficient performance for ultimate home comfort."
}
```


## 8️⃣ Context Window Demonstration

Large context increases token usage and may impact accuracy.

In [ ]:
long_context = """The iPhone 17 redefines the standard for Apple’s flagship lineup, bridging the gap between its base and Pro models. It features a stunning 6.3-inch Super Retina XDR OLED display that finally introduces 120Hz ProMotion technology to the base model, providing ultra-smooth scrolling and an Always-On experience. With a peak outdoor brightness of 3,000 nits, it ensures perfect visibility even in direct sunlight.
Under the hood, the A19 chip (built on a 3nm process) delivers massive performance gains, including a 50% faster CPU and up to 90% faster GPU compared to its predecessor. This power drives Apple Intelligence, enabling on-device tools like Live Translation, Writing Tools, and advanced Clean Up photo editing.
The camera system sees a major evolution with a 48MP Dual Fusion setup, combining a high-resolution main lens with a new 48MP Ultra Wide sensor. A standout addition is the 18MP Center Stage front camera, which uses a square sensor to automatically frame group shots and allow high-res landscape selfies without rotating the phone. Available in vibrant colors like Lavender, Mist Blue, and Sage, the iPhone 17 is protected by Ceramic Shield 2, offering three times better scratch resistance.
"""
prompt = long_context + "\nSummarize in 2 lines."
response = model.generate_content(prompt)

print("Long text\n")
print(long_context)

print("Summarized text\n")
print(response.text)

Long text

The iPhone 17 redefines the standard for Apple’s flagship lineup, bridging the gap between its base and Pro models. It features a stunning 6.3-inch Super Retina XDR OLED display that finally introduces 120Hz ProMotion technology to the base model, providing ultra-smooth scrolling and an Always-On experience. With a peak outdoor brightness of 3,000 nits, it ensures perfect visibility even in direct sunlight.
Under the hood, the A19 chip (built on a 3nm process) delivers massive performance gains, including a 50% faster CPU and up to 90% faster GPU compared to its predecessor. This power drives Apple Intelligence, enabling on-device tools like Live Translation, Writing Tools, and advanced Clean Up photo editing.
The camera system sees a major evolution with a 48MP Dual Fusion setup, combining a high-resolution main lens with a new 48MP Ultra Wide sensor. A standout addition is the 18MP Center Stage front camera, which uses a square sensor to automatically frame group shots and

## 9️⃣ Deterministic Output (Temperature = 0)

Setting temperature to 0 produces consistent output.

In [ ]:
for i in range(3):
    response = model.generate_content(
        "Explain Magento in one sentence.",
        generation_config={"temperature": 0}
    )
    print(response.text)
    print("---")

Magento is a robust e-commerce platform that empowers businesses to build and manage highly customizable and scalable online stores.
---
Magento is a robust e-commerce platform that empowers businesses to build and manage highly customizable and scalable online stores.
---
Magento is a robust e-commerce platform that empowers businesses to build and manage highly customizable and scalable online stores.
---
